# Actividad 1: Pronóstico de Temperatura Diaria (Serie Univariante)
Comparación de arquitecturas recurrentes: **SimpleRNN**, **LSTM** y **GRU**.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

URL_DATASET = (
    "https://raw.githubusercontent.com/jbrownlee/Datasets/"
    "master/daily-min-temperatures.csv"
)

try:
    df = pd.read_csv(URL_DATASET)
    df["Date"] = pd.to_datetime(df["Date"])
    df["Temp"] = pd.to_numeric(df["Temp"], errors="coerce")
    df = df.dropna(subset=["Date", "Temp"]).sort_values("Date").reset_index(drop=True)
    fuente_datos = "Daily Minimum Temperatures (dataset público)"
except Exception as e:
    print(f"Aviso: no se pudo descargar el CSV externo ({e}). Usando serie de respaldo.")
    fechas = pd.date_range(start="2018-01-01", periods=1460, freq="D")
    dias = np.arange(len(fechas))
    estacionalidad = 13.5 + 5.5 * np.sin(2 * np.pi * dias / 365.25 - np.pi / 2)
    ruido = np.random.normal(0, 1.4, size=len(fechas))
    temp = np.round(estacionalidad + ruido, 1)
    df = pd.DataFrame({"Date": fechas, "Temp": temp})
    fuente_datos = "Serie climática diaria simulada (respaldo local)"

df = df.set_index("Date")
print(f"Fuente: {fuente_datos}")
print(f"Registros: {len(df)} | Rango: {df.index.min().date()} a {df.index.max().date()}")
print(df.describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df.index, df["Temp"], color="tab:blue", linewidth=1)
axes[0].set_title("Serie Temporal Diaria de Temperatura")
axes[0].set_xlabel("Fecha")
axes[0].set_ylabel("Temperatura (°C)")

axes[1].hist(df["Temp"], bins=30, color="tab:cyan", edgecolor="black", alpha=0.8)
axes[1].set_title("Distribución de Temperatura")
axes[1].set_xlabel("Temperatura (°C)")
axes[1].set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

LOOK_BACK = 14
TEST_RATIO = 0.2

values = df[["Temp"]].values.astype("float32")
split_idx = int(len(values) * (1 - TEST_RATIO))

scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(values[:split_idx])
test_scaled = scaler.transform(values[split_idx - LOOK_BACK:])

def crear_ventanas(serie_2d, look_back=14):
    X, y = [], []
    for i in range(look_back, len(serie_2d)):
        X.append(serie_2d[i - look_back:i, 0])
        y.append(serie_2d[i, 0])
    return np.array(X, dtype="float32")[..., np.newaxis], np.array(y, dtype="float32")

X_train, y_train = crear_ventanas(train_scaled, LOOK_BACK)
X_test, y_test = crear_ventanas(test_scaled, LOOK_BACK)

def construir_modelo(arquitectura, input_shape, unidades=64):
    model = Sequential(name=f"Modelo_{arquitectura}")
    if arquitectura == "SimpleRNN":
        model.add(SimpleRNN(unidades, activation="tanh", input_shape=input_shape))
    elif arquitectura == "LSTM":
        model.add(LSTM(unidades, activation="tanh", input_shape=input_shape))
    elif arquitectura == "GRU":
        model.add(GRU(unidades, activation="tanh", input_shape=input_shape))
    else:
        raise ValueError("Arquitectura no soportada")
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

arquitecturas = ["SimpleRNN", "LSTM", "GRU"]
historiales = {}
predicciones_inv = {}
metricas = []

y_test_real = scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
fechas_test = df.index[split_idx:]
early_stop = EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)

for arq in arquitecturas:
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    modelo = construir_modelo(arq, input_shape=(LOOK_BACK, 1), unidades=64)
    hist = modelo.fit(
        X_train,
        y_train,
        validation_split=0.15,
        epochs=35,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0,
    )
    historiales[arq] = hist.history

    y_pred_scaled = modelo.predict(X_test, verbose=0)
    y_pred = scaler.inverse_transform(y_pred_scaled).ravel()
    predicciones_inv[arq] = y_pred

    mae = mean_absolute_error(y_test_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_real, y_pred))
    mape = np.mean(np.abs((y_test_real - y_pred) / np.clip(np.abs(y_test_real), 1e-5, None))) * 100

    metricas.append({
        "Modelo": arq,
        "MAE (°C)": round(mae, 3),
        "RMSE (°C)": round(rmse, 3),
        "MAPE (%)": round(mape, 2),
        "Épocas": len(hist.history["loss"]),
    })

df_resultados = pd.DataFrame(metricas).sort_values("RMSE (°C)")
print("\n=== Comparación de Modelos en Test ===")
print(df_resultados.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for arq in arquitecturas:
    axes[0].plot(historiales[arq]["loss"], label=f"{arq} - Train")
    axes[0].plot(historiales[arq]["val_loss"], linestyle="--", label=f"{arq} - Val")
axes[0].set_title("Curvas de Pérdida (MSE)")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("MSE")
axes[0].legend(fontsize=8)

axes[1].plot(fechas_test, y_test_real, label="Real", color="black", linewidth=1.8)
for arq in arquitecturas:
    axes[1].plot(fechas_test, predicciones_inv[arq], label=arq, alpha=0.85, linewidth=1.2)
axes[1].set_title("Pronóstico de Temperatura en Conjunto de Prueba")
axes[1].set_xlabel("Fecha")
axes[1].set_ylabel("Temperatura (°C)")
axes[1].legend()

plt.tight_layout()
plt.show()